In [5]:
import madina as md
import madina.una.tools as una
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely import Point, MultiPoint, LineString, Polygon
import requests
import json
import time


##### data from func

In [3]:
berlin = md.Zonal()

berlin.load_layer(
    name='sidewalks',
    source='/workspaces/exp_sidewalk_planning/data/berlin/sidewalks_pseudo.geojson'
)

berlin.load_layer('buildings', '/workspaces/exp_sidewalk_planning/data/berlin/buildings_pseudo.geojson')
berlin.load_layer('kitas', '/workspaces/exp_sidewalk_planning/data/berlin/kitas_pseudo.geojson')
berlin.describe()

berlin.create_street_network(source_layer="sidewalks", node_snapping_tolerance=0.1)

berlin.insert_node(label='origin', layer_name="kitas")
berlin.insert_node(label='destination', layer_name="buildings")

berlin.create_graph()

Layer name           | Visible | projection | rows  | File path           
sidewalks            |       1 | EPSG:3857  |   357 | /workspaces/exp_sidewalk_planning/data/berlin/sidewalks_pseudo.geojson
buildings            |       1 | EPSG:3857  |  2860 | /workspaces/exp_sidewalk_planning/data/berlin/buildings_pseudo.geojson
kitas                |       1 | EPSG:3857  |    80 | /workspaces/exp_sidewalk_planning/data/berlin/kitas_pseudo.geojson
Geographic center: (13.396303813961305, 52.4916423315232)
No network graph yet. First, insert a layer that contains network segments (streets, sidewalks, ..) and call create_street_network(layer_name,  weight_attribute=None)
	Then,  insert origins and destinations using 'insert_nodes(label, layer_name, weight_attribute)'
	Finally, when done, create a network by calling 'create_street_network()'


In [32]:
def betweeness(origin:str, destination:str, job_id:int, search_radius: int,detour_ratio:float = 1.2, knn_plateau:int = 400, beta:float = 0.001):
    
    berlin.clear_nodes()
    berlin.insert_node(layer_name=origin, label='origin')
    berlin.insert_node(layer_name=destination, label='destination')
    berlin.create_graph()

    una.betweenness(
        berlin,
        search_radius = search_radius,
        detour_ratio=detour_ratio,
        decay=True,
        decay_method="exponent",
        beta=beta,
        turn_penalty=True,
        closest_destination=False,
        elastic_weight = True,
        knn_weight = [0.5, 0.25, 0.25],
        knn_plateau = 400,
        save_betweenness_as="betweenness",
        num_cores=8
    )
    
    gdf = berlin['sidewalks'].gdf
    cols = ['strassenna','bezirk', 'geometry', 'weight', 'betweenness']
    gdf = gdf[cols]
       
    return gdf

In [53]:
test_func = betweeness(job_id=1, origin= "buildings", destination= "kitas", search_radius= 200, beta= 0.001, knn_plateau= 10)

In [57]:
test_func.geometry.crs

<Projected CRS: EPSG:3857>
Name: WGS 84 / Pseudo-Mercator
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: World between 85.06°S and 85.06°N.
- bounds: (-180.0, -85.06, 180.0, 85.06)
Coordinate Operation:
- name: Popular Visualisation Pseudo-Mercator
- method: Popular Visualisation Pseudo Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

#### data from the api

In [6]:
def execute_process(origin:str, destination:str, search_radius: int, detour_ratio:float = 1.2, knn_plateau:int = 400, beta:float = 0.001):

  url = "http://host.docker.internal:5000/api/model/processes/betweeness/execution"

  payload = json.dumps({
    "inputs": {
      "origin": origin,
      "destination": destination,
      "search_radius": search_radius,
      "detour_ratio":detour_ratio,
      "knn_plateau": knn_plateau,
      "beta": beta
      
    },
    "outputs": {},
    "response": "raw",
    "subscriber": {}
  })
  headers = {
    'Content-Type': 'application/json'
  }

  response = requests.request("POST", url, headers=headers, data=payload)
  job_id = response.json().get( 'job_id')
  return job_id

In [7]:
def get_results(job_id):
    
    url = f"http://host.docker.internal:5000/api/model/jobs/{job_id}/results"

    payload={}
    headers = {}

    response = requests.request("GET", url, headers=headers, data=payload)
    data = response.json()
    gdf = gpd.GeoDataFrame.from_features(data['result']['features'])
    return gdf


In [52]:
###### increased process time extermely because of calling func before process finished.
##### wait_for_completion does not work so far

"""def execute_betweeness(origin:str, destination:str, search_radius: int, detour_ratio:float = 1.2, knn_plateau:int = 400, beta:float = 0.001):
    job_id = execute_process(origin, destination, search_radius,knn_plateau, beta)
    wait_for_completion(job_id)
    result_gdf = get_results(job_id)
    return result_gdf"""

'def execute_betweeness(origin:str, destination:str, search_radius: int, detour_ratio:float = 1.2, knn_plateau:int = 400, beta:float = 0.001):\n    job_id = execute_process(origin, destination, search_radius,knn_plateau, beta)\n    wait_for_completion(job_id)\n    result_gdf = get_results(job_id)\n    return result_gdf'

In [14]:
test_api = execute_process(origin= "Wohnhaus", destination= "Kindergarten", search_radius= 200, beta= 0.001, knn_plateau= 10, detour_ratio = 1.3)

In [12]:
test_api = get_results(test_api)

In [13]:
test_api

,geometry,strassenna,bezirk,weight,betweenness
0,"LINESTRING (1492149.161 6890174.449, 1492239.9...",Baerwaldstraße,Friedrichshain-Kreuzberg,212.854732,0.0
1,"LINESTRING (1490509.829 6890309.573, 1490535.4...",Blücherstraße,Friedrichshain-Kreuzberg,135.229074,0.0
2,"LINESTRING (1490643.841 6890294.319, 1490653.9...",Blücherstraße,Friedrichshain-Kreuzberg,496.847035,0.0
3,"LINESTRING (1490787.212 6890666.151, 1490817.0...",Gitschiner Straße,Friedrichshain-Kreuzberg,436.880588,0.0
4,"LINESTRING (1492015.822 6890743.524, 1492404.6...",Gitschiner Straße,Friedrichshain-Kreuzberg,402.118597,0.0
...,...,...,...,...,...
942,"LINESTRING (1495541.235 6887907.471, 1495557.3...",Fuldastraße,Neukölln,216.485153,0.0
943,"LINESTRING (1496052.729 6889042.642, 1496068.0...",Weichselpark,Neukölln,210.738348,0.0
944,"LINESTRING (1494736.864 6888287.913, 1494736.3...",Mainzer Straße,Neukölln,32.084393,0.0
945,"LINESTRING (1494889.064 6888174.538, 1494891.1...",Reuterstraße,Neukölln,34.353264,0.0
